In [0]:
%run ./00_setup

In [0]:
# fill shelves and inventories based on the new or existing orders and order items. Then update order and robots records
from pyspark.sql import Window
import pyspark.sql.functions as F
import uuid


def generate_shelves_inventory_data():
    # 1. Get detailed order items with weights
    # We need weights at the item level to calculate cumulative sum accurately
    order_item_details = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
        .filter("status='Initial' AND is_current=true")
        .join(spark.table("workspace.amazon_fullfilment_silver.order_items_silver"), "order_id")
        .join(spark.table("workspace.amazon_fullfilment_silver.products_silver").filter("is_current=true"), "product_id")
        .select("order_id", "product_id", "quantity", "customer_id", "orderdate", (F.col("quantity") * F.col("weight_kg")).alias("item_total_weight"))
    )

    # 2. Calculate Cumulative Weight to assign "Shelf Groups"
    # We group items until they hit the 30kg threshold
    running_weight_window = Window.orderBy("order_id", "product_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    # assign_idx determines which "30kg bucket" the item falls into
    order_buckets = order_item_details.withColumn("cumulative_weight", F.sum("item_total_weight").over(running_weight_window)) \
        .withColumn("shelf_rank", F.floor(F.col("cumulative_weight") / 30.0).cast("int"))

    # 3. Get Available Shelves and Robots (Unique pairs)
    # Join Robots to their Active Shelves first
    shelf_pool = (spark.table("workspace.amazon_fullfilment_silver.shelves")
                  .filter("is_active = true AND status = 'Idle'")
                  .alias("shlf")
                  .join(spark.table("workspace.amazon_fullfilment_silver.robots").alias("rb"), "robot_id")
                  .filter("rb.is_active = true AND  rb.status = 'Idle' AND rb.current_battery_pct > 10")
                  # Ensure we only get the latest event per robot
                  .withColumn("rn", F.row_number().over(Window.partitionBy("robot_id").orderBy(F.col("rb.Updated_Timestamp").desc())))
                  .filter("rn = 1")
                  .drop("rn")
                  .withColumn("available_shelf_rank", F.row_number().over(Window.orderBy(F.col("current_battery_pct").desc())))
                  # Adjust rank to start at 0 to match our F.floor index
                  .withColumn("available_shelf_rank", F.col("available_shelf_rank") - 1)
    )

    # 4. Final Assignment Join
    # Match the "Shelf Group" (0, 1, 2...) with the "Available Shelf Rank"
    assignments = order_buckets.join(shelf_pool, F.col("shelf_rank") == F.col("available_shelf_rank"), "inner")

    # --- CREATE OUTPUT DATAFRAMES ---

    # A. Shelves (Now unique per shelf_id)
    shelves_df = assignments.select(
        "shelf_id", "robot_id", "shlf.max_weight_capacity",
        F.lit("Picking").alias("status"),
        F.current_timestamp().alias("updated_timestamp")
    ).distinct()

    # B. Inventory (Multiple items can now share the same shelf_id)
    inventory_df = assignments.select(
        "shelf_id", "product_id", "quantity", "order_id"
    )

    # C. Robot Updates
    robot_updates_df = assignments.select(
        "robot_id",
        F.lit("Picking").alias("status"),
        F.lit(500.0).alias("max_weight_capacity"),
        "current_battery_pct",
        F.current_timestamp().alias("Updated_Timestamp")
    ).distinct()

    # D. Order Status Updates
    current_run_uuid = str(uuid.uuid4())

    order_status_updates_df = assignments.select(
        "order_id",
        "customer_id",
        "orderdate",
        F.lit("Picking").alias("status"),
        F.current_timestamp().alias("updated_timestamp")
    ).distinct()

    return shelves_df, inventory_df, robot_updates_df, order_status_updates_df

# Execute and Save
shelves_df, inventory_df, robot_updates_df, order_status_updates_df = generate_shelves_inventory_data()
display(shelves_df)
display(inventory_df)
display(robot_updates_df)
display(order_status_updates_df)

# save to the source volume

(shelves_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/shelves_events"))
(inventory_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/inventory"))
(robot_updates_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/robot_events"))
(order_status_updates_df.write
.format("csv")
.option("header",True)
.option("delimiter",",")
.mode("append")
.save(f"{source_volume_path}/orders"))